# Evaluation workflow demonstration

In [1]:
import wilson_suite as ws

from wilson_suite.wilson_main.wf import WilsonSimulation
# from wilson_suite.wilson_main.workflow_abstractions import WilsonSimulation

from CQCParse.utils import PKG_ROOT as CQCPARSE_ROOT

import numpy as np

## Experiment and possible spectral axes

In [2]:
evv_experiment_obj = ws.fixtures.evv_experiment()
evv_experiment_obj # can there be a better way to show details of an experiment?

VibExperiment(order=3, field=ElectricField(pulses=(EmPulse(env='impulsive', maxstr=1e-05, tc=50.0, cf=0.0, cf_uv=0.0, dev=None, wv=[0.0, 0.0, 1.0], pol=[1.0, 0.0, 0.0], overall_phase=(1+0j), id=1), EmPulse(env='impulsive', maxstr=1e-05, tc=100.0, cf=0.0, cf_uv=0.0, dev=None, wv=[0.0, 0.0, 1.0], pol=[1.0, 0.0, 0.0], overall_phase=(1+0j), id=2), EmPulse(env='impulsive', maxstr=1e-05, tc=120.0, cf=0.0, cf_uv=0.072, dev=None, wv=[0.0, 0.0, 1.0], pol=[1.0, 0.0, 0.0], overall_phase=(1+0j), id=3))), detector=SpecDetector(detection_method='freq', detector_location=(0.0, 0.0, 1.0), detection_polarization=(1.0, 0.0, 0.0), detection_range=[0.003, 0.0031, 0.0032, 0.0033, 0.0034000000000000002, 0.0035, 0.0036, 0.0037, 0.0038, 0.0039000000000000003, 0.004, 0.0041, 0.004200000000000001, 0.0043, 0.0044, 0.0045000000000000005, 0.0046, 0.0047, 0.0048000000000000004, 0.0049, 0.005, 0.0051, 0.0052, 0.0053, 0.0054, 0.0055, 0.005600000000000001, 0.0057, 0.0058, 0.005900000000000001, 0.006, 0.0061, 0.0062000

In [3]:
help(evv_experiment_obj) # no info on indep_vars or valid_axis_combs

Help on VibExperiment in module wilson_suite.wilson_experiment.abstractions object:

class VibExperiment(builtins.object)
 |  VibExperiment(order: int, field: wilson_suite.wilson_experiment.abstractions.ElectricField, detector: wilson_suite.wilson_experiment.abstractions.SpecDetector, scans: list[wilson_suite.wilson_experiment.abstractions.SpecScan] = None, magn_conditions: list = <factory>) -> None
 |
 |  Class to represent a vibrational wave-mixing experiment
 |
 |  ----
 |  field: ElectricField instance: A "base" perturbing field (upon which scans may be imposed)
 |  detector: SpecDetector instance: The detector for this experiment
 |  scans: List of SpecScan instances: Tells which parameters will be scanned over (and how) in this experiment
 |  magn_conditions: List of lists [[sign*pulse i (is always of lower frequency than...), sign*pulse j], ...]:
 |  Magnitude conditions for later use in identifying terms that will not become fully resononant in this experiment
 |
 |  Methods de

In [4]:
evv_experiment_obj.indep_vars # hmmmm, what is the meaning of this dictionary

{0: [[(-1,), (2,)]]}

In [5]:
evv_experiment_obj.valid_axis_combs # hmmmm, what is the meaning of this dictionary

{((-1,), (2,)): [{'A': [(-1,)], 'B': [(-1,), (2,)]},
  {'A': [(-1,)], 'B': [(2,)]},
  {'A': [(2,)], 'B': [(-1,)]},
  {'A': [(2,)], 'B': [(-1,), (2,)]}]}

In [6]:
axes_choice = evv_experiment_obj.valid_axis_combs[((-1,), (2,))][3]
axes_choice

{'A': [(2,)], 'B': [(-1,), (2,)]}

## Derived terms

In [7]:
terms = ws.derive.main.get_fully_enhanced_terms(experiment=evv_experiment_obj)
terms # hmmmm, what is the meaning of this dictionary
#  0: {(0, 0): []} -- doesn't make sense to have? also, will it ever be not empty?

{1: {(1,
   0): [VibPerturbedTerm(coeff = -1/4, props = [PolProp(ops = [QOperator(o = 1, op_type = None, ax = None)], dord = 1, (inds = ['a'])), PolProp(ops = [QOperator(o = 2, op_type = None, ax = None)], dord = 1, (inds = ['b'])), PolProp(ops = [QOperator(o = 0, op_type = None, ax = None), QOperator(o = 3, op_type = None, ax = None)], dord = 2, (inds = ['a', 'b']))], freqterms = [[sl=omega_['a'], sr=omega_[]], pert_wf=F, [sl=omega_['b'], sr=omega_[]], pert_wf=F], res = [ResCond(diff = [sl=omega_[], sr=omega_['a']], pert_wf=F, pf = [-1], id = None), ResCond(diff = [sl=omega_['b'], sr=omega_['a']], pert_wf=F, pf = [-1, 2], id = None)]), VibPerturbedTerm(coeff = -1/4, props = [PolProp(ops = [QOperator(o = 0, op_type = None, ax = None), QOperator(o = 3, op_type = None, ax = None)], dord = 1, (inds = ['b'])), PolProp(ops = [QOperator(o = 1, op_type = None, ax = None)], dord = 1, (inds = ['a'])), PolProp(ops = [QOperator(o = 2, op_type = None, ax = None)], dord = 2, (inds = ['a', 'b']))], 

In [8]:
evv_terms = ws.derive.term_var_translate.translate_terms_to_axis_variables(terms, axes_choice)
evv_terms
#  0: {(0, 0): []} -- doesn't make sense to have? also, will it ever be not empty?

{1: {(1,
   0): [VibPerturbedTerm(coeff = -1/4, props = [PolProp(ops = [QOperator(o = 1, op_type = None, ax = None)], dord = 1, (inds = ['a'])), PolProp(ops = [QOperator(o = 2, op_type = None, ax = None)], dord = 1, (inds = ['b'])), PolProp(ops = [QOperator(o = 0, op_type = None, ax = None), QOperator(o = 3, op_type = None, ax = None)], dord = 2, (inds = ['a', 'b']))], freqterms = [[sl=omega_['a'], sr=omega_[]], pert_wf=F, [sl=omega_['b'], sr=omega_[]], pert_wf=F], res = [ResCond(diff = [sl=omega_[], sr=omega_['a']], pert_wf=F, pf = ['-A', 'B'], id = None), ResCond(diff = [sl=omega_['b'], sr=omega_['a']], pert_wf=F, pf = ['B'], id = None)]), VibPerturbedTerm(coeff = -1/4, props = [PolProp(ops = [QOperator(o = 0, op_type = None, ax = None), QOperator(o = 3, op_type = None, ax = None)], dord = 1, (inds = ['b'])), PolProp(ops = [QOperator(o = 1, op_type = None, ax = None)], dord = 1, (inds = ['a'])), PolProp(ops = [QOperator(o = 2, op_type = None, ax = None)], dord = 2, (inds = ['a', 'b']

## Calculation/simulation setup

In [9]:
mol_system = ws.main.abstractions.MolecularSystem(name='FORM', natoms=4)
vib_ana = ws.main.abstractions.VibAnaSetup(system=mol_system, regime='GVPT2', vibana_own_analysis='none')
calc_setup = ws.main.abstractions.DataOriginInfo(source_type='gaussian', 
                                                    lvl_theory='B3LYP', 
                                                    basis_set='cc-pVQZ', 
                                                    base_file_loc=CQCPARSE_ROOT+'/CQCParse/files_examples/dftGaussian/FORM/B3LYPcc_pVQZ/g16_inputFull_3q.out')

In [10]:
help(vib_ana)

Help on VibAnaSetup in module wilson_suite.wilson_main.abstractions object:

class VibAnaSetup(builtins.object)
 |  VibAnaSetup(regime: str = None, system: wilson_suite.wilson_main.abstractions.MolecularSystem = None, regime_subinfo: dict = None, vibana_own_analysis: str = 'full', max_state_lvl: int = None, states: list[wilson_suite.wilson_main.abstractions.VibState] = None, nc_sqrt_eigval: dict = None, nc_eigvec: dict = None, exclude_modes: list = None, diagn: dict = None) -> None
 |
 |  Class for setup for vibrational analysis and storage of the resulting information
 |
 |      ----
 |      regime: string: Vibrational analysis regime (e.g. "harmonic", "GVPT2", "VPT2")
 |      system: MolecularSystem instance: System to which this instance pertains
 |      regime_subinfo: dictionary: Extra configuration info for vibrational regime (e.g. skip rotational effects)
 |      max_state_lvl: integer: Maximum number of vibrational quanta per harmonic state involved in states
 |      states: Li

In [11]:
bounds_dict = {'A': (0., 5000.), 'B': (0., 5000.)}
box = ws.intensities.amplitudes.spectrum_composition.Box(bounds_dict)
spectral_window = ws.intensities.amplitudes.spectrum_composition.SpectralWindow(box=box)

evi = ws.main.spectrum_abstractions.EvaluationInfo(**{'spectral_window': spectral_window,
                                                        'Gamma': 4.7, 'Gamma_unit': 'cm-1',
                                                        'grid_resolution': {'A': 10, 'B': 10}})
spec_eval_setup = ws.main.spectrum_abstractions.SpecEvalSetup(ev_info=evi)

### Making a `WilsonSimulation` instance

In [12]:
simulation = WilsonSimulation()
simulation.terms = evv_terms
simulation.system = mol_system
simulation.exp = evv_experiment_obj
simulation.vib_ana_setup = vib_ana
simulation.spec_eval_setup = spec_eval_setup

#### Preparing `simulation.props`

In [13]:
simulation.addPropEvalSetup(eval_uniform=calc_setup)
simulation.setPropsAndMaxStateLvl() # setting up self.props/sim.props
simulation.dressPropsWithSetup()

In [14]:
simulation.getResults(obtainer=ws.utils.wilson_data_obtainer)

data_request dict_keys(['dipgrad', 'polhess', 'polgrad', 'diphess', 'cff', 'nc_sqrt_eigval', 'anharmonic_states'])


In [15]:
print(simulation.is_ready)
print(simulation.is_configured)

True
True


In [16]:
import copy
copy_props = copy.deepcopy(simulation.props)

simulation.props = []
print(simulation.is_ready)

simulation.props = copy_props

not len(self.props) > 0
False


In [17]:
simulation.vib_ana_setup.states

[VibState(harm_quanta_coeffs={('0',): 1.0}, energy=2726.813, displacement=None, serial_harm_quanta_coeffs={'0': 1.0}, state_label='0', harmonic_WF=None),
 VibState(harm_quanta_coeffs={('1',): 1.0}, energy=1794.54, displacement=None, serial_harm_quanta_coeffs={'1': 1.0}, state_label='1', harmonic_WF=None),
 VibState(harm_quanta_coeffs={('2',): 1.0}, energy=1501.586, displacement=None, serial_harm_quanta_coeffs={'2': 1.0}, state_label='2', harmonic_WF=None),
 VibState(harm_quanta_coeffs={('3',): 1.0}, energy=1185.288, displacement=None, serial_harm_quanta_coeffs={'3': 1.0}, state_label='3', harmonic_WF=None),
 VibState(harm_quanta_coeffs={('4',): 1.0}, energy=2682.765, displacement=None, serial_harm_quanta_coeffs={'4': 1.0}, state_label='4', harmonic_WF=None),
 VibState(harm_quanta_coeffs={('5',): 1.0}, energy=1247.878, displacement=None, serial_harm_quanta_coeffs={'5': 1.0}, state_label='5', harmonic_WF=None),
 VibState(harm_quanta_coeffs={('0', '0'): 1.0}, energy=5390.225, displacement

#### Evaluate


```python
    def evaluate(self, keep_intermediates):
        workflow = EvaluationWorkflow(self)
        self._workflow = workflow

        self.spec, info = workflow.run(keep_intermediates)

        if self.diagn is None:
            self.diagn = {}
        self.diagn.update(info)
```

In [18]:
# hides all evaluation workflow
simulation.evaluate(keep_intermediates=True)

In [19]:
assert 'timing' in simulation.diagn
assert 'intermediates' in simulation.diagn  # Now included!
assert len(simulation.diagn['intermediates']['prep_terms']) == 14
assert len(simulation.diagn['intermediates']['all_features']) == 60
assert len(simulation.diagn['intermediates']) == 11  # All steps

np.set_printoptions(linewidth=280, precision=3)

print(simulation.diagn.keys())

dict_keys(['timing', 'total_time', 'intermediates'])


## `EvaluationWorkflow` using set `WilsonSimulation`

In [24]:
workflow = ws.intensities.amplitudes.evaluation_wf.EvaluationWorkflow(simulation)
results = workflow.run(keep_intermediates=True)

```python
    def run(self, *, keep_intermediates: bool = False):
        """Run evaluation, return (spectrum, info_dict)"""
        self._validate_inputs()
        start = time.time()
        
        try:
            # prep steps
            terms_list = self._step('prep_terms', self._prep_terms)
            vib_data, vib_cache, data_configs = self._step('prep_data', self._prep_data)

            # get resonances locations for all terms
            motif_locs, terms_for_motifs = self._step('process_resonances', 
                lambda: self._process_resonances(terms_list, vib_data, vib_cache))
            
            coefficients = self._step('term_coefficients',
                lambda: self._calc_coefficients(terms_list, motif_locs, data_configs))
            
            features = self._step('all_features',
                lambda: self._extract_features(motif_locs, terms_for_motifs, coefficients))
            
            spec_window = self._step('place_in_specwindow',
                lambda: self._place_in_specwindow(features))

            grid_manager = self._step('make_GridManager',
                lambda: self._make_GridManager(spec_window))
            grid_manager.make_fullgrid(self.simulation.spec_eval_setup.ev_info.grid_resolution)
            
            regions = self._step('make_regions',
                lambda: self._make_regions(grid_manager))
            
            self._step('prep_complilers',
                lambda: self._prep_complilers(vib_data, vib_cache, self.simulation.spec_eval_setup.ev_info.Gamma))
            
            regions_results = self._step('evaluate_regions',
                lambda: self._evaluate_regions(regions))
            
            self._step('assemble_fullgrid',
                lambda: self._assemble_fullgrid(grid_manager, regions_results))

    
            info = {'timing': self.timing, 'total_time': time.time() - start}
            if keep_intermediates:
                info['intermediates'] = self.results
            
            return grid_manager.full_grid, info
            
        except Exception as e:
            # On error, always include intermediates
            info = {
                'timing': self.timing,
                'total_time': time.time() - start,
                'failed_at': self.failed_at,
                'error': str(e),
                'intermediates': self.results,
            }

            raise type(e)(f"Failed at '{self.failed_at}': {e}") from e
```

In [25]:
spec_info, wf_info = results

In [26]:
spec_info.keys()

dict_keys(['A', 'B', 'result'])

In [27]:
wf_info.keys()

dict_keys(['timing', 'total_time', 'intermediates'])

In [30]:
wf_info['intermediates'].keys(), len(wf_info['intermediates'].keys())

(dict_keys(['prep_terms', 'prep_data', 'process_resonances', 'term_coefficients', 'all_features', 'place_in_specwindow', 'make_GridManager', 'make_regions', 'prep_complilers', 'evaluate_regions', 'assemble_fullgrid']),
 11)

## Steps of the evaluation workflow:

- Step 0: `self._validate_inputs()` - check if `WilsonSimulation` has everything needed for the evaluation.
- Step 1: `prep_terms` - make a flat list of `VibPerturbedTerm` from `WilsonSimulation.terms`.
- Step 2: `prep_data` 
  - 1) prepare "inclusion list" of normal mode labels 
  - 2) make an instances of `VibStatesData`, `VibDiffCache`, `MolPropsCollection`, and `EvaluationDataAndConfigs`
- Step 3: `process_resonances` - makes 2 mappings:
   - `motif_res_loc` (`ResLocGeoObject`(value) to `ResonanceMotif`(key))
   - `terms_for_motifs` (`list['VibPerturbedTerm']`(value) to `ResonanceMotif`(key)) 
- Step 4: `term_coefficients` 
   - 1) identify what to precalculate and precalculate 
   - 2) make a mapping of a collection of info `dict[ParameterSet, 'apl_coeff as float']]` to  `'VibPerturbedTerm'`   
- Step 5: `all_features` - make a `list[SpectralFeature]` for all res.locations for given terms
- Step 6: `place_in_specwindow` - identify `SpectralFeature`s within `SpectralWindow` and the ones contributing to it (based on dynamic range and lineshape parameter - #TODO)
- Step 7: `make_GridManager` - make a `GridManager` instance and also `make_fullgrid()` so there is an attribute `GridManager.full_grid`
- Step 8: `make_regions` - make  `GridRegion`s from the full grid based on features clustering:
  - 1) Use `spec_window.find_clusters_by_featboxes()` to make clusters of features based on the overlap of the `SpectralFeature.feat_box` which by now should be set (based on dynamic range and lineshape parameter - #TODO)
  - 2) Cut `full_grid` into subgrids 
  - 3) register these subgrids into `GridRegion`s
- Step 9: `prep_complilers` - prepare another layer of convenient data strucure: `PhysicsCalculator` and `FeatureCompiler`. They prepare numerical terms (in particular, the motif part) from symbolic ones
  - `FeatureCompiler` takes one `SpectralFeature` and goes over its `SpectralFeature.term_contributions` to get each terms' res_motif
  - `PhysicsCalculator` can `evaluate_compiled_group()` [and `evaluate_resonance_motif()` for a motif in group] - essentially computes final resonance denominator (1/rescond1/rescond2/...)
- Step 10: `evaluate_regions` - the last layer is `_evaluate_feature()` where the amplitude coefficient is multiplied by the grid evaluated res_motifs (`self.physics.evaluate_compiled_group(group, coords)` for a group per different res_motif found in `SpectralFeature.term_contributions`)
  - 1) for region in regions - `_evaluate_region()`
  - 2) for feature in region.features - `_evaluate_feature()`
  - 3) for compiled_group - `evaluate_compiled_group()`
- Step 11: `assemble_fullgrid` - takes `regions_results` and places them according to indices information for each region into the right places back in the `full_grid`



prep_terms
[VibPerturbedTerm(coeff = -1/4, props = [PolProp(ops = [QOperator(o = 1, op_type = None, ax = None)], dord = 1, (inds = ['a'])), PolProp(ops = [QOperator(o = 2, op_type = None, ax = None)], dord = 1, (inds = ['b'])), PolProp(ops = [QOperator(o = 0, op_type = None, ax = None), QOperator(o = 3, op_type = None, ax = None)], dord = 2, (inds = ['a', 'b']))], freqterms = [[sl=omega_['a'], sr=omega_[]], pert_wf=F, [sl=omega_['b'], sr=omega_[]], pert_wf=F], res = [ResCond(diff = [sl=omega_[], sr=omega_['a']], pert_wf=F, pf = ['-A', 'B'], id = None), ResCond(diff = [sl=omega_['b'], sr=omega_['a']], pert_wf=F, pf = ['B'], id = None)]), VibPerturbedTerm(coeff = -1/4, props = [PolProp(ops = [QOperator(o = 0, op_type = None, ax = None), QOperator(o = 3, op_type = None, ax = None)], dord = 1, (inds = ['b'])), PolProp(ops = [QOperator(o = 1, op_type = None, ax = None)], dord = 1, (inds = ['a'])), PolProp(ops = [QOperator(o = 2, op_type = None, ax = None)], dord = 2, (inds = ['a', 'b']))], 